<a href="https://colab.research.google.com/github/SavageLDN/refill-labels/blob/main/refill_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import io
import pandas as pd
import barcode
from barcode.writer import ImageWriter

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# 1. Dataset
data = [
    {"Item": "Dish Soap Refill", "Batch_ID": "DS-2026-001", "Expiry_Date": "2027-08-18", "Volume_ml": 500},
    {"Item": "Hand Wash Refill", "Batch_ID": "HW-2026-002", "Expiry_Date": "2027-08-18", "Volume_ml": 250},
    {"Item": "Surface Cleaner", "Batch_ID": "SC-2026-003", "Expiry_Date": "2028-01-01", "Volume_ml": 1000},
    {"Item": "Laundry Detergent", "Batch_ID": "LD-2026-004", "Expiry_Date": "2027-11-30", "Volume_ml": 1500},
]
df = pd.DataFrame(data)

# 2. Function to generate a barcode image buffer in memory
def generate_barcode_buffer(code_text):
    code128 = barcode.get_barcode_class('code128')
    rv = io.BytesIO()
    barcode_instance = code128(code_text, writer=ImageWriter())
    # Generate barcode graphic without built-in text overlay to prevent double-printing
    barcode_instance.write(rv, options={"write_text": False, "quiet_zone": 1.0, "module_height": 10.0})
    rv.seek(0);
    return rv

# 3. Typography and styles
styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'LabelTitle',
    parent=styles['Normal'],
    fontSize=10,
    leading=12,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor("#111111"),
    spaceAfter=3
)
meta_style = ParagraphStyle(
    'LabelMeta',
    parent=styles['Normal'],
    fontSize=8,
    leading=10,
    fontName='Helvetica',
    textColor=colors.HexColor("#333333"),
    spaceAfter=4
)
batch_style = ParagraphStyle(
    'LabelBatch',
    parent=styles['Normal'],
    fontSize=7,
    leading=9,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor("#555555"),
    alignment=1 # Centred
)

# 4. Build individual, structured label cards
label_cards = []
for _, row in df.iterrows():
    barcode_buf = generate_barcode_buffer(row['Batch_ID'])
    # Barcode image with explicit height and width
    barcode_img = RLImage(barcode_buf, width=170, height=28)

    card_elements = [
        Paragraph(row['Item'], title_style),
        Paragraph(f"Vol: {row['Volume_ml']} ml &nbsp;|&nbsp; Exp: {row['Expiry_Date']}", meta_style),
        Spacer(1, 4),
        barcode_img,
        Spacer(1, 2),
        Paragraph(f"BATCH: {row['Batch_ID']}", batch_style)
    ]

    # Place elements into a bordered label container
    card_table = Table([[card_elements]], colWidths=[240])
    card_table.setStyle(TableStyle([
        ('BOX', (0, 0), (-1, -1), 0.75, colors.HexColor("#b0b0b0")),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('LEFTPADDING', (0, 0), (-1, -1), 10),
        ('RIGHTPADDING', (0, 0), (-1, -1), 10),
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor("#ffffff")),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ]))
    label_cards.append(card_table)

# 5. Arrange into a 2-column printable A4 grid
grid_data = []
for i in range(0, len(label_cards), 2):
    row_cells = [label_cards[i]]
    if i + 1 < len(label_cards):
        row_cells.append(label_cards[i + 1])
    else:
        row_cells.append("")
    grid_data.append(row_cells)

layout_table = Table(grid_data, colWidths=[255, 255])
layout_table.setStyle(TableStyle([
    ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 14),
]))

# 6. Build PDF document
pdf_filename = "refill_labels.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=A4,
    leftMargin=20,
    rightMargin=20,
    topMargin=25,
    bottomMargin=25
)
doc.build([layout_table])
print(f"Generated clean {pdf_filename} with structured layout.")

Generated clean refill_labels.pdf with structured layout.


In [9]:
pip install reportlab

In [10]:
pip install python-barcode

In [11]:
pip install pillow

In [16]:
from google.colab import files
files.download('refill_labels.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
files.download('refill_labels.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import io
import pandas as pd
import barcode
from barcode.writer import ImageWriter

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from google.colab import files

# 1. Direct Home Refill Item List (Edit or add your items directly here)
home_items = [
    {"Item": "Eco Dish Soap", "Refill_Code": "HOME-DS-01", "Category": "Kitchen", "Capacity": "500 ml", "Directions": "Ready to use. 1-2 pumps per wash."},
    {"Item": "Anti-Bacterial Hand Wash", "Refill_Code": "HOME-HW-02", "Category": "Bathroom", "Capacity": "250 ml", "Directions": "Refill into foaming pump dispenser."},
    {"Item": "Multi-Surface Cleaner", "Refill_Code": "HOME-SC-03", "Category": "General Household", "Capacity": "750 ml", "Directions": "Dilute 50ml into warm spray bottle."},
    {"Item": "Non-Bio Laundry Liquid", "Refill_Code": "HOME-LL-04", "Category": "Laundry", "Capacity": "1500 ml", "Directions": "Use 30ml per standard 5kg load."},
    {"Item": "White Vinegar & Citrus", "Refill_Code": "HOME-WV-05", "Category": "Cleaning Spray", "Capacity": "500 ml", "Directions": "Shake well before spraying on glass."},
    {"Item": "Bicarbonate of Soda", "Refill_Code": "HOME-BS-06", "Category": "Pantry Refill", "Capacity": "1000 g", "Directions": "Store in a cool, dry airtight container."}
]

df = pd.DataFrame(home_items)

# 2. Barcode Generator
def generate_barcode_buffer(code_text):
    code128 = barcode.get_barcode_class('code128')
    rv = io.BytesIO()
    barcode_instance = code128(code_text, writer=ImageWriter())
    barcode_instance.write(rv, options={"write_text": False, "quiet_zone": 1.0, "module_height": 9.0})
    rv.seek(0)
    return rv

# 3. Text Styles
styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'HomeTitle',
    parent=styles['Normal'],
    fontSize=11,
    leading=13,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor("#1a1a1a"),
    spaceAfter=2
)
meta_style = ParagraphStyle(
    'HomeMeta',
    parent=styles['Normal'],
    fontSize=8,
    leading=10,
    fontName='Helvetica',
    textColor=colors.HexColor("#4a4a4a"),
    spaceAfter=3
)
direct_style = ParagraphStyle(
    'HomeDirections',
    parent=styles['Normal'],
    fontSize=7.5,
    leading=9.5,
    fontName='Helvetica-Oblique',
    textColor=colors.HexColor("#666666"),
    spaceAfter=2
)
code_style = ParagraphStyle(
    'HomeCode',
    parent=styles['Normal'],
    fontSize=7,
    leading=8,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor("#333333"),
    alignment=1
)

# 4. Label Card Dimensions (85mm x 50mm - 2 columns per A4 page)
CARD_WIDTH = 88 * mm
CARD_HEIGHT = 48 * mm

label_cards = []
for _, row in df.iterrows():
    barcode_buf = generate_barcode_buffer(row['Refill_Code'])
    barcode_img = RLImage(barcode_buf, width=55 * mm, height=10 * mm)

    card_elements = [
        Paragraph(row['Item'], title_style),
        Paragraph(f"Category: <b>{row['Category']}</b> &nbsp;|&nbsp; Refill Size: <b>{row['Capacity']}</b>", meta_style),
        Paragraph(row['Directions'], direct_style),
        Spacer(1, 1.5 * mm),
        barcode_img,
        Spacer(1, 0.5 * mm),
        Paragraph(row['Refill_Code'], code_style)
    ]

    # Border styled as light scissor cut guidelines for matt sticky paper
    card_table = Table([[card_elements]], colWidths=[CARD_WIDTH], rowHeights=[CARD_HEIGHT])
    card_table.setStyle(TableStyle([
        ('BOX', (0, 0), (-1, -1), 0.5, colors.HexColor("#cccccc")),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
        ('LEFTPADDING', (0, 0), (-1, -1), 7),
        ('RIGHTPADDING', (0, 0), (-1, -1), 7),
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor("#ffffff")),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ]))
    label_cards.append(card_table)

# 5. Grid Layout
grid_data = []
for i in range(0, len(label_cards), 2):
    row_cells = [label_cards[i]]
    if i + 1 < len(label_cards):
        row_cells.append(label_cards[i + 1])
    else:
        row_cells.append("")
    grid_data.append(row_cells)

layout_table = Table(grid_data, colWidths=[92 * mm, 92 * mm])
layout_table.setStyle(TableStyle([
    ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 5 * mm),
]))

# 6. Build PDF and trigger download
pdf_filename = "home_refill_labels.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=A4,
    leftMargin=12 * mm,
    rightMargin=12 * mm,
    topMargin=15 * mm,
    bottomMargin=15 * mm
)
doc.build([layout_table])

print("PDF successfully generated. Downloading now...")
files.download(pdf_filename)

PDF successfully generated. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>